In [1]:
import sys
import argparse
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import numpy as np
import anndata
import statsmodels.api as sm
import statsmodels.formula.api as smf
import tqdm
import pandas as pd



import anndata
import csv
import gzip
import os
import scipy.io
import numpy as np 
import anndata

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import scanpy
from matplotlib.pyplot import rc_context




In [3]:

# --- INIT ---

# which data to use
head_folder = "/".join(os.getcwd().split("/")[:3])
data_head_folder = '%s/Dropbox/aorta_circadian_data/datasets/joint' % head_folder
hvg_to_use = 'prior_knowledge_guided'

# get corresponding paths
adata_path = '%s/adata_qc_filtered.h5ad' % data_head_folder
hvg_folder = '%s/data_annotations/hvg/%s' % (data_head_folder,hvg_to_use)
scvi_res_folder = '%s/scvi_res' % hvg_folder
scvi_mean_embedding_df_path = '%s/scvi_mean_embedding.tsv' % scvi_res_folder
umap_path_out = '%s/scvi_mean_umap_embedding.tsv' % scvi_res_folder
cluster_resolution = 0.05
clustering_folder = '%s/clustering/res_%s' % (scvi_res_folder, str(cluster_resolution))
cluster_df_fileout = '%s/clusters.tsv' % (clustering_folder)


# # SMC subclusters
# clustering_folder = '%s/Dropbox/aorta_circadian_data/datasets/joint/data_annotations/hvg/transformed_X_outlier_variance/scvi_res/clustering/res_0.05/subclustering/res_0.5' % head_folder
# cluster_df_fileout = '%s/smc_subclusters.tsv' % (clustering_folder)



# settings
min_prop = 1e-7




In [4]:
# --- LOAD ADATA ---

adata = anndata.read_h5ad(adata_path)

adata


AnnData object with n_obs × n_vars = 145271 × 32285
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2'

In [5]:
# --- ADD EMBEDDINGS TO ADATA ---



# ** load clusters **
cluster_df = pd.read_table(cluster_df_fileout,sep='\t',index_col='index')

# ** make sure everything in the same order
adata = adata[list(cluster_df.index)]


# ** add embeddings **
adata.obs["cluster"] = np.array(cluster_df['leiden_scvi_cluster'])
# adata.obs["cluster"] = np.array(cluster_df['smc_subcluster'])

adata


/var/folders/y2/0zd17m8j6bz5j5xv0cy2dw680000gp/T/ipykernel_1669/1264531170.py:13: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["cluster"] = np.array(cluster_df['leiden_scvi_cluster'])


AnnData object with n_obs × n_vars = 145271 × 32285
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description', 'cluster'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2'

In [6]:
# --- GET THE UNIQUE DESCRIPTION AND CLUSTERS ---

descriptions = list(adata.obs['description'].unique())
clusters = sorted(list(adata.obs['cluster'].unique()))
clusters = list(filter(lambda x: ~np.isnan(x),clusters)) # get rid of NaN cluster (only relevant for SMC subcluster)
clusters = list(map(lambda x: int(x),clusters))


print("Unique descriptions:\n",descriptions)
print("Unique clusters:\n",clusters)


# --- LIMIT ADATA TO GENES THAT MEET MINIMUM PSEUDOBULK COUNT THRESHOLD ---


# ** get gene pseudobulk counts
adata.var['pseudobulk_count'] = np.array(np.sum(adata.X,axis=0)).flatten()

# ** get the number of cells in the smallest cell type
smallest_cell_type_adata = adata[adata.obs['cluster'] == np.max(clusters)]

# ** get pseudobulk cutoff **
pseudobulk_threshold = min_prop * np.sum(smallest_cell_type_adata.obs['lib_size'])

# ** limit adata to this **
adata = adata[:,adata.var['pseudobulk_count'] >= pseudobulk_threshold]


adata


Unique descriptions:
 ['male aligned bmal1-ko', 'male misaligned bmal1-control', 'female aligned bmal1-ko', 'female misaligned bmal1-control', 'male aligned bmal1-control', 'female aligned bmal1-control']
Unique clusters:
 [0, 1, 2, 3, 4, 5, 6, 7, 8]


View of AnnData object with n_obs × n_vars = 145271 × 26120
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description', 'cluster'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2', 'pseudobulk_count'

In [7]:
# --- IDENTIFY GENES MEETING THE MINIMUM PROP IN EACH CLUSTER / CONDITION COMBO ---

import warnings
warnings.filterwarnings('ignore')

genes_to_est = set()
for i, description in enumerate(descriptions):
    for j, cluster in enumerate(clusters):
        print("Description: %s; Cluster: %s" % (i,j))
            
        cluster_condition_adata = adata[(adata.obs['cluster'] == cluster) & (adata.obs['description'] == description)]
        cluster_condition_adata.var['prop'] = np.array(np.sum(cluster_condition_adata.X,axis=0)).flatten()  / np.sum(cluster_condition_adata.obs['lib_size'])
        cluster_condition_adata = cluster_condition_adata[:,cluster_condition_adata.var['prop'] >= min_prop]
        genes_to_est.update(list(cluster_condition_adata.var_names))

# make it a list
genes_to_est = list(genes_to_est)



Description: 0; Cluster: 0
Description: 0; Cluster: 1
Description: 0; Cluster: 2
Description: 0; Cluster: 3
Description: 0; Cluster: 4
Description: 0; Cluster: 5
Description: 0; Cluster: 6
Description: 0; Cluster: 7
Description: 0; Cluster: 8
Description: 1; Cluster: 0
Description: 1; Cluster: 1
Description: 1; Cluster: 2
Description: 1; Cluster: 3
Description: 1; Cluster: 4
Description: 1; Cluster: 5
Description: 1; Cluster: 6
Description: 1; Cluster: 7
Description: 1; Cluster: 8
Description: 2; Cluster: 0
Description: 2; Cluster: 1
Description: 2; Cluster: 2
Description: 2; Cluster: 3
Description: 2; Cluster: 4
Description: 2; Cluster: 5
Description: 2; Cluster: 6
Description: 2; Cluster: 7
Description: 2; Cluster: 8
Description: 3; Cluster: 0
Description: 3; Cluster: 1
Description: 3; Cluster: 2
Description: 3; Cluster: 3
Description: 3; Cluster: 4
Description: 3; Cluster: 5
Description: 3; Cluster: 6
Description: 3; Cluster: 7
Description: 3; Cluster: 8
Description: 4; Cluster: 0
D

In [8]:
len(genes_to_est)

22167

In [9]:
# old_genes_to_est_path = '/Users/benauerbach/Dropbox/aorta_circadian_data/datasets/joint/data_annotations/hvg/transformed_X_outlier_variance/scvi_res/clustering/res_0.05/nonparametric_reg_old/genes_to_est.txt'
# with open(old_genes_to_est_path) as file_obj:
#     old_genes_to_est = list(map(lambda x: x.replace("\n",""), file_obj.readlines()))

    
# new_genes_to_est = list(set(genes_to_est).difference(set(old_genes_to_est)))

# len(new_genes_to_est)


In [10]:
# --- MAKE THE FOLDER OUTS ---

# head folder
reg_head_folder = '%s/nonparametric_reg' % clustering_folder
if not os.path.exists(reg_head_folder):
    os.makedirs(reg_head_folder)


# subfolders for each cluster / condition combo
cluster_description_folder_out_dict = {}
for cluster in clusters:
    cluster_reg_folder = '%s/cluster_%s' % (reg_head_folder,cluster)
    if not os.path.exists(cluster_reg_folder):
        os.makedirs(cluster_reg_folder)
    for description in descriptions:
        cluster_description_reg_folder = '%s/%s' % (cluster_reg_folder,description)
        if not os.path.exists(cluster_description_reg_folder):
            os.makedirs(cluster_description_reg_folder)
            
        # update dict
        if cluster not in cluster_description_folder_out_dict:
            cluster_description_folder_out_dict[cluster] = {}
        cluster_description_folder_out_dict[cluster][description] = cluster_description_reg_folder

    
    

In [11]:
# --- WRITE OUT THE GENES TO ESTIMATE ---

genes_to_est_fileout = '%s/genes_to_est.txt' % reg_head_folder
with open(genes_to_est_fileout,"wb") as file_obj:
    file_obj.write("\n".join(genes_to_est).encode())




In [12]:
# --- ADD PHASE COL ---

adata.obs['phase'] = np.array((adata.obs['zt'] / 24.0) * 2 * np.pi)



In [13]:
raise Exception("ONLY USING ENDOTHELIAL AND MANUALLY SETTING NB COEF!")



Exception: FLIPPING THE ORDER OF THE CLUSTER REGRESSINS!

In [13]:
# --- FIT ---


import torch
import tempo2
from tempo2 import identify_de_novo_cyclers

failed_list = []
# for cluster in clusters:
for cluster in [2]:
    for description in descriptions:
        
        try:
        
            print("----------------")
            print("Cluster %s; Description %s" % (cluster,description))


            # ** get cluster, condition adata **
            cluster_condition_adata = adata[(adata.obs['cluster'] == cluster) & (adata.obs['description'] == description)]
            cluster_condition_adata = cluster_condition_adata[:,genes_to_est]
            # cluster_condition_adata = cluster_condition_adata[:,new_genes_to_est]
            # cluster_condition_adata = cluster_condition_adata[:,['Dbp','Arntl','Myh11']]

            
            # ** prep **
            cluster_condition_adata.var['prop'] = np.array(np.sum(cluster_condition_adata.X,axis=0)).flatten() / np.sum(cluster_condition_adata.obs['lib_size'])
            if 'log_L' not in adata.obs:
                cluster_condition_adata.obs['log_L'] = np.array(np.log(cluster_condition_adata.obs['lib_size']))
            cluster_condition_adata = cluster_condition_adata[:,cluster_condition_adata.var['prop'] > 0]

            
            # 
            # ** run **
            tempo2.identify_de_novo_cyclers.run(adata = cluster_condition_adata,
                folder_out = cluster_description_folder_out_dict[cluster][description],
                num_grid_points = 4,
                phases_sampled = torch.Tensor(np.array(cluster_condition_adata.obs['phase'])).unsqueeze(1), # torch.Tensor(np.array(adata.obs['phase'])).unsqueeze(1),
                cell_phase_dist = None, # cell_posterior_obj, # cell_posterior_obj # cell_posterior_obj
                use_nb = True,
                log_mean_log_disp_coef= torch.Tensor(np.array([-3.001958370208740234e+00, -1.134198158979415894e-01])),# None, torch.Tensor(np.array([-3.001958370208740234e+00, -1.134198158979415894e-01]))
                lr = 1e-1,
                num_waveform_est_cell_samples = 1,
                num_waveform_est_gene_samples = 1,
                num_bf_est_cell_samples = 5,
                num_bf_est_gene_samples = 5,
                vi_max_epochs = 300,
                vi_print_epoch_loss = True,
                vi_improvement_window = 5,
                vi_convergence_criterion = 1e-3,
                cosinor_zero_frac_num_samples_per_waveform = 100,
                num_waveform_bf_cell_samples = 5,
                num_waveform_bf_gene_samples = 5,
                )


            
            
            
 
            
        except Exception as e:
            print(e)
            
            failed = '%s_%s' % (cluster,description)
            failed_list.append(failed)
            
            
            
            


----------------
Cluster 2; Description male aligned bmal1-ko
Iter: 0; ELBO 462.28: E[LL]: -462.28; KL: 0.0
Iter: 1; ELBO 453.779: E[LL]: -453.726; KL: 0.053
Iter: 2; ELBO 446.467: E[LL]: -446.297; KL: 0.171
Iter: 3; ELBO 440.262: E[LL]: -439.936; KL: 0.327
Iter: 4; ELBO 434.856: E[LL]: -434.355; KL: 0.501
Iter: 5; ELBO 430.097: E[LL]: -429.418; KL: 0.679
Iter: 6; ELBO 426.03: E[LL]: -425.177; KL: 0.853
Iter: 7; ELBO 421.564: E[LL]: -420.545; KL: 1.019
Iter: 8; ELBO 417.405: E[LL]: -416.233; KL: 1.172
Iter: 9; ELBO 413.677: E[LL]: -412.365; KL: 1.313
Iter: 10; ELBO 409.973: E[LL]: -408.532; KL: 1.441
Improvement: 0.061980378177115325
Iter: 11; ELBO 406.495: E[LL]: -404.935; KL: 1.56
Improvement: 0.05721056014646253
Iter: 12; ELBO 403.131: E[LL]: -401.46; KL: 1.671
Improvement: 0.05396001119779148
Iter: 13; ELBO 399.885: E[LL]: -398.107; KL: 1.778
Improvement: 0.05140813519183518
Iter: 14; ELBO 396.795: E[LL]: -394.91; KL: 1.885
Improvement: 0.04929186863463508
Iter: 15; ELBO 394.026: E

Iter: 49; ELBO 152.643: E[LL]: -147.216; KL: 5.427
Improvement: 0.0013634332275539451
Iter: 50; ELBO 152.568: E[LL]: -147.109; KL: 5.459
Improvement: 0.001212726812084619
Iter: 51; ELBO 152.561: E[LL]: -147.075; KL: 5.486
Improvement: 0.0014250819345081878
Iter: 52; ELBO 152.665: E[LL]: -147.158; KL: 5.508
Improvement: 0.0012485748858970247
Iter: 53; ELBO 152.549: E[LL]: -147.023; KL: 5.526
Improvement: 0.0011551278292215583
Iter: 54; ELBO 152.537: E[LL]: -146.996; KL: 5.541
Improvement: 0.0010128037753656116
Iter: 55; ELBO 152.546: E[LL]: -146.994; KL: 5.552
Improvement: 0.0008881221256454852
Loss converged and stopping
----------------
Cluster 2; Description female aligned bmal1-ko
Iter: 0; ELBO 535.588: E[LL]: -535.588; KL: 0.0
Iter: 1; ELBO 524.909: E[LL]: -524.857; KL: 0.052
Iter: 2; ELBO 516.455: E[LL]: -516.285; KL: 0.17
Iter: 3; ELBO 509.399: E[LL]: -509.072; KL: 0.327
Iter: 4; ELBO 502.885: E[LL]: -502.384; KL: 0.501
Iter: 5; ELBO 497.358: E[LL]: -496.68; KL: 0.677
Iter: 6; EL

Iter: 38; ELBO 158.352: E[LL]: -153.554; KL: 4.798
Improvement: 0.005682964409430502
Iter: 39; ELBO 158.25: E[LL]: -153.347; KL: 4.903
Improvement: 0.00509234393848379
Iter: 40; ELBO 158.202: E[LL]: -153.203; KL: 4.999
Improvement: 0.004594762629932969
Iter: 41; ELBO 158.207: E[LL]: -153.121; KL: 5.086
Improvement: 0.003983152143481661
Iter: 42; ELBO 158.121: E[LL]: -152.956; KL: 5.165
Improvement: 0.0036627246072415964
Iter: 43; ELBO 157.935: E[LL]: -152.698; KL: 5.237
Improvement: 0.003445505117507297
Iter: 44; ELBO 158.013: E[LL]: -152.711; KL: 5.302
Improvement: 0.003004110694547535
Iter: 45; ELBO 157.956: E[LL]: -152.595; KL: 5.361
Improvement: 0.002540337186469288
Iter: 46; ELBO 157.953: E[LL]: -152.539; KL: 5.414
Improvement: 0.0021841654283309975
Iter: 47; ELBO 157.918: E[LL]: -152.456; KL: 5.462
Improvement: 0.0020081912707327065
Iter: 48; ELBO 157.867: E[LL]: -152.362; KL: 5.504
Improvement: 0.0018090934630162758
Iter: 49; ELBO 157.933: E[LL]: -152.391; KL: 5.542
Improvement:

Iter: 32; ELBO 307.729: E[LL]: -303.177; KL: 4.552
Improvement: 0.014052913331620176
Iter: 33; ELBO 307.267: E[LL]: -302.515; KL: 4.751
Improvement: 0.012761660858368606
Iter: 34; ELBO 306.783: E[LL]: -301.837; KL: 4.945
Improvement: 0.011665234001021818
Iter: 35; ELBO 306.699: E[LL]: -301.566; KL: 5.133
Improvement: 0.010443609292385192
Iter: 36; ELBO 306.24: E[LL]: -300.928; KL: 5.312
Improvement: 0.009438688235956283
Iter: 37; ELBO 305.955: E[LL]: -300.471; KL: 5.484
Improvement: 0.008631504006723079
Iter: 38; ELBO 305.683: E[LL]: -300.037; KL: 5.647
Improvement: 0.007791723757883284
Iter: 39; ELBO 305.579: E[LL]: -299.778; KL: 5.8
Improvement: 0.006914143876604162
Iter: 40; ELBO 305.312: E[LL]: -299.367; KL: 5.946
Improvement: 0.006047786548846856
Iter: 41; ELBO 305.032: E[LL]: -298.949; KL: 6.082
Improvement: 0.00560887790113096
Iter: 42; ELBO 304.831: E[LL]: -298.62; KL: 6.211
Improvement: 0.005032514105780206
Iter: 43; ELBO 304.881: E[LL]: -298.548; KL: 6.333
Improvement: 0.0044

In [14]:
cluster_description_folder_out_dict[cluster][description]

'/Users/mingyaolab/Dropbox/aorta_circadian_data/datasets/joint/data_annotations/hvg/prior_knowledge_guided/scvi_res/clustering/res_0.05/nonparametric_reg/cluster_2/female aligned bmal1-control'